<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_0515_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 주제

Orbit Wars 환경에서 행성을 점령하는 실시간 전략(RTS) 게임 AI 개발

# 데이터
Orbit Wars 게임 환경에서 매 턴 제공되는 observation 데이터 사용

사용된 주요 변수

- planets: 행성 정보

- step: 현재 게임 턴

- player_id: 현재 플레이어 번호

- owner_abs: 각 행성의 소유자

- ships: 각 행성에 있는 병력 수

- alive: 행성 활성 상태

- planet_prod: 행성의 생산력

- f_owner, f_ships: 이동 중인 함대의 소유자와 병력 수


# 코드흐름

In [ ]:
# 게임 상태를 tensor 형태로 변환
obs_tensors=single_obs_to_tensor(obs,player_id=player_id)

# observation을 파싱하여 현재 행성, 병력, 소유자 정보를 읽음
obs=pars_obs(obs_tensors)

# 행성 이동 정보와 거리 정보를 계산
movement=ensure_planet_movement(...)
cache=build_distance_cache(movement, max_k=int(config.horizon))

# 공격 가능한 목표 후보를 생성
target_idx, target_exists=build_target_shortlist(...)

# 각 공격 후보의 점수를 계산
score=score_candidates(...)

# 점수가 높은 후보부터 Greedy 방식으로 공격 선택
wave_entries, leftover=_greedy_select(...)

# 남은 병력은 regroup 전략으로 재배치
regroup_entries=_plan_regroup(...)

# 최종 행동을 Kaggle 제출 형식으로 변환
return_sparse_action_row_to_moves(sparse_row, obs, plater_id=player_id)

# 새롭게 알게 된 점 / 어려운 내용 / 배울 점

parse_obs(obs_tensors): 게임 환경에서 제공되는 observation 데이터를 해석하여 행성 정보, 병력 정보, 소유자 정보 등으로 구조화하는 함수

원래 데이터가
obs_tensors=[
  [1, 0, 50, 30,
  [2, 1, 30, 5]
  ]
와 같이 숫자 덩어리로 들어오면
obs=parse_obs(obs_tensors)를 통해
obs.owner
obs.ships
obs.production
와 같이 의미 있는 이름을 붙여준다.

- enwure_planet_movement: 행성 간 이동 정보(planet movement)가 존재하는지 확인하고 없으면 생성

- cache=캐시: 컴퓨터에서 계산한 결과를 저장해두는 임시 저장소

ex) 행성이 100개이면, 행성 간 거리 계산ㅇ르 매번 하면 매우 느리기 때문에, 한 번 계산해놓고 cache[(1,2)]=7과 같이 저장

- build_distance_cache(...): 행성 간 거리 정보를 미리 계산해서 저장하는 함수

- max_k=30: 30턴 뒤까지 거리 계산

(최대 몇 단계까지 볼 건지)

- config.horizon: 머신러닝/강화학습/게임 AI에서 앞으로 얼마나 먼 미래까지 볼 것인가

config.horizon=20

: 현재 턴 뿐만 아니라 20턴 뒤까지 예상하면서 행동 결정.

- max_k=int(config.horizon): distance ache를 만들 때는 반드시 정수가 필요하기 때문에, int를 통해 정수로 바꿔줌.

- _greedy_select(): 현재 가장 좋아 보이는 선택부터 하나씩 고르는 방법

: 공격 후보들의 점수를 비교한 뒤 가장 효율적인 공격부터 선택하는 Greedy 알고리즘 수행

sparse_row: 실제로 행동이 발생한 부분만 저장한 압축 형태의 행동 데이터

: 실제 행동이 있는 부분만 기록하여 메모리, 연산량을 줄임.

행동 공간(action space이 엄청 큰 게임에서는 대부분의 행동이 사실상 "0"(아무것도 안 함) 상태.

- 일반적인 머신러닝 프로젝트는 csv 데이터를 한 번 읽고 학습하지만, Orbit Wars는 매 턴마다 게임 상태가 계속 변함.

- _greedy_select()는 가장 점수가 높은 공격을 선택하는 함수인데, 단순 정렬이 아니라 병력 제한과 남은 자원까지 고려해야 함.



